# Building Hybrid Search — Part 1: The Lexical Leg

*Ingesting a real PDF and searching it by keyword, using Db2 Text Search (backed by OpenSearch).*

---

**Hybrid search** combines two complementary ways of finding text:

| Leg | Matches by | Strong on |
|-----|------------|-----------|
| **Lexical** &nbsp;*(this notebook)* | exact keywords / tokens | precise terms — API names, error codes, identifiers |
| **Semantic** &nbsp;*(later)* | meaning, via embeddings | paraphrases, synonyms, "what I meant" |

Production systems run **both** and *fuse* the rankings. Here we build the **lexical leg end-to-end** on a real document, so every moving part is concrete before we add vectors.

### The pipeline we'll walk through

```mermaid
flowchart TB
    PDF["📄 PDF"] -->|Docling| MD["📝 Markdown"]
    MD -->|HybridChunker| CH["📦 Chunks"]
    CH -->|INSERT| TBL[("🗄️ Db2 table · pdf_chunks")]
    TBL -->|"db2ts: create · activate · populate"| IDX["🔎 Db2 Text Search index"]
    IDX <-->|"lexical engine"| OS[("🟧 OpenSearch")]
    Q(["🔍 CONTAINS / SCORE — SQL"]) -->|query| IDX
    IDX -->|"ranked chunks + relevance"| RES(["📊 Top-N results"])

    classDef store fill:#e0f2fe,stroke:#0284c7,color:#075985;
    classDef engine fill:#ffedd5,stroke:#ea580c,color:#9a3412;
    class TBL,IDX store;
    class OS engine;
```

> **Scope:** this is the *lexical half* of hybrid search — a working foundation, not the finished system. We close with exactly what's left to add.

In [14]:
# --- Setup: config, credentials, helpers -------------------------------------
import os, sys, subprocess, textwrap
import ibm_db
import pandas as pd

PDF_PATH    = "LLM_Integration_Doc.pdf"
MD_PATH     = "LLM_Integration_Doc.md"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # the chunk token-budget tracks this model
MAX_TOKENS  = 256
SCHEMA, TABLE, INDEX_NAME = "myschema", "pdf_chunks", "pdf_chunks_idx"

# Db2 credentials come from .env (DB2_PASSWORD, ...). Never hard-code secrets.
def _load_env(path=".env"):
    if os.path.exists(path):
        for line in open(path):
            s = line.strip()
            if s and not s.startswith("#") and "=" in s:
                k, _, v = s.partition("="); os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
_load_env()

DSN = (f"DATABASE={os.environ.get('DB2_DATABASE','sample')};"
       f"HOSTNAME={os.environ.get('DB2_HOST','localhost')};"
       f"PORT={os.environ.get('DB2_PORT','50000')};PROTOCOL=TCPIP;"
       f"UID={os.environ.get('DB2_USER','db2inst1')};PWD={os.environ['DB2_PASSWORD']};")

def db2_connect():
    return ibm_db.connect(DSN, "", "")

def query_df(conn, sql):
    "Run a SELECT and return rows as a tidy DataFrame."
    st = ibm_db.exec_immediate(conn, sql)
    rows, r = [], ibm_db.fetch_assoc(st)
    while r:
        rows.append(dict(r)); r = ibm_db.fetch_assoc(st)
    return pd.DataFrame(rows)

def run_as_db2inst1(script):
    """Db2 Text Search admin goes through db2ts, which must run locally as the
    instance owner — not over the notebook's remote SQL connection. So we invoke
    the deployed shell script as db2inst1 (passwordless sudo is configured here)."""
    out = subprocess.run(["sudo","-niu","db2inst1","bash","-lc",f"/home/db2inst1/{script}"],
                         capture_output=True, text=True)
    print((out.stdout or out.stderr).strip())
    return out.returncode

print("config loaded — database:", os.environ.get("DB2_DATABASE","sample"))

config loaded — database: sample


## Step 1 — Extract: PDF ➜ Markdown with Docling

A PDF is *layout*, not clean text. **Docling** parses the document's real structure — headings, paragraphs, tables, lists — and emits **Markdown**, preserving the hierarchy we'll exploit when chunking.

Extraction is the slow part and you do it once per document, so it lives in its own script, `pdf_to_markdown.py`.

In [15]:
# Run the extractor (skips if the .md is already present).
if not os.path.exists(MD_PATH):
    print(subprocess.run([sys.executable, "pdf_to_markdown.py"], capture_output=True, text=True).stdout)

markdown = open(MD_PATH).read()
print(f"{MD_PATH}: {len(markdown):,} characters\n")
print(textwrap.indent("\n".join(markdown.splitlines()[:18]), "  "))

LLM_Integration_Doc.md: 40,254 characters

  ## Chapter 10. Large language model integration with Db2

  In Db2, you can now register external watsonx.ai models directly within the database, enabling seamless integration of AI capabilities into your data workflows. To manage these external models, Db2 introduces new DDL statements to register models, remove them, and control usage privileges. By using the built-in TO\_EMBEDDING function, you can generate vector embeddings from text data, making it easier to do advanced analytics, such as semantic search or similarity comparisons. Furthermore, by using the built-in TEXT\_GENERATION function, you can generate text from a given prompt using a registered text generation model. Additionally, metadata can be tracked through the catalog views SYSCAT.EXTERNALMODELS, SYSCAT.EXTERNALMODELOPTIONS, and SYSCAT.EXTERNALMODELAUTH.

  ## Background and proposed solution

  Db2 12.1.2 introduced support for a native VECTOR data type and a set of associ

## Step 2 — Chunk: Docling **HybridChunker**

Retrieval works on *chunks*, not whole documents — and the chunking strategy is a real quality lever. Too big and the relevant sentence gets diluted; too small and you lose context.

We use Docling's **HybridChunker**, the strategy most RAG practitioners reach for:

- **Structure-aware** — follows the document's headings/sections instead of cutting blindly every *N* characters.
- **Token-aware** — merges chunks that are too small and splits chunks that exceed a token budget.

We size that budget to the embedding model the *semantic* leg will use later (`all-MiniLM-L6-v2`, 256 tokens), so each chunk is already embedding-ready.

In [16]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

doc = DocumentConverter().convert(MD_PATH).document
chunker = HybridChunker(tokenizer=HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL), max_tokens=MAX_TOKENS))

# contextualize() prepends each chunk's heading trail, so the chunk carries its context.
chunks = [(" > ".join(c.meta.headings or []), chunker.contextualize(chunk=c))
          for c in chunker.chunk(dl_doc=doc)]
print(f"{len(chunks)} chunks\n")

pd.DataFrame([{"chunk_id": i, "headings": h, "preview": t[:80].replace(chr(10)," ") + "…"}
              for i, (h, t) in enumerate(chunks[:5], start=1)])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (729 > 512). Running this sequence through the model will result in indexing errors


101 chunks



,chunk_id,headings,preview
0,1,Chapter 10. Large language model integration w...,Chapter 10. Large language model integration w...
1,2,Background and proposed solution,Background and proposed solution Db2 12.1.2 in...
2,3,Background and proposed solution,Background and proposed solution The features ...
3,4,CREATE EXTERNAL MODEL statement,CREATE EXTERNAL MODEL statement The CREATE EXT...
4,5,model-name,model-name Specifies the external model. The n...


## Step 3 — Store the chunks in Db2

Each chunk becomes a row in `myschema.pdf_chunks`, keyed by `chunk_id`. This table is the **source of truth**; the search index in Step 4 is built from it.

We rebuild from scratch on every run. A Db2 table can't be dropped while a text index sits on it — and that index is owned by `db2ts` (run as `db2inst1`) — so we drop the index first, then rebuild the table.

In [17]:
# Clean slate: drop the text index first (as db2inst1) so the table can be rebuilt.
run_as_db2inst1("drop_pdf_chunks_index.sh")

conn = db2_connect()
try:    ibm_db.exec_immediate(conn, f"DROP TABLE {SCHEMA}.{TABLE}")
except Exception: pass
ibm_db.exec_immediate(conn, f"""
    CREATE TABLE {SCHEMA}.{TABLE} (
        chunk_id   INTEGER NOT NULL PRIMARY KEY,
        headings   VARCHAR(1024),
        chunk_text CLOB(1M))""")

ins = ibm_db.prepare(conn, f"INSERT INTO {SCHEMA}.{TABLE} (chunk_id, headings, chunk_text) VALUES (?,?,?)")
for cid, (h, t) in enumerate(chunks, start=1):
    ibm_db.bind_param(ins,1,cid); ibm_db.bind_param(ins,2,h); ibm_db.bind_param(ins,3,t)
    ibm_db.execute(ins)

print(query_df(conn, f"SELECT COUNT(*) CHUNKS, MIN(chunk_id) LO, MAX(chunk_id) HI FROM {SCHEMA}.{TABLE}").to_string(index=False), "\n")
query_df(conn, f"SELECT chunk_id, headings FROM {SCHEMA}.{TABLE} ORDER BY chunk_id FETCH FIRST 5 ROWS ONLY")

==> Connecting to 'sample'

   Database Connection Information

 Database server        = DB2/LINUXX8664 12.1.5.0
 SQL authorization ID   = DB2INST1
 Local database alias   = SAMPLE


==> Dropping text index myschema.pdf_chunks_idx (if it exists)
CIE00001 Operation completed successfully. 
    dropped.

Done. You can now re-run chunk_and_ingest.ipynb, then ./index_pdf_chunks.sh.
 CHUNKS  LO  HI
    101   1 101 



,CHUNK_ID,HEADINGS
0,1,Chapter 10. Large language model integration w...
1,2,Background and proposed solution
2,3,Background and proposed solution
3,4,CREATE EXTERNAL MODEL statement
4,5,model-name


## Step 4 — Build the lexical index (Db2 Text Search ➜ OpenSearch)

This is the integration point. We create a **Db2 Text Search index** over `chunk_text`. Behind the scenes Db2 hands the text to **OpenSearch**, which builds the inverted (lexical) index — but we never call OpenSearch directly. Db2 manages it, and we'll query through plain SQL.

The index is built with `db2ts` as `db2inst1`: **create ➜ activate ➜ populate**. The script also waits for OpenSearch to make the documents searchable before returning.

In [18]:
run_as_db2inst1("index_pdf_chunks.sh")

==> Step 1: Connecting to 'sample'

   Database Connection Information

 Database server        = DB2/LINUXX8664 12.1.5.0
 SQL authorization ID   = DB2INST1
 Local database alias   = SAMPLE


==> Step 2: Dropping the existing text index (if any)

==> Step 3: Finding the registered OpenSearch server
    OpenSearch server id = 7

==> Step 4: Building the text index on myschema.pdf_chunks(chunk_text)
CIE00001 Operation completed successfully. 
CIE00001 Operation completed successfully. 
CIE00001 Operation completed successfully. 

==> Step 5: Test search — chunks mentioning 'embedding'
    matches for 'embedding': 10

CHUNK_ID    HEADINGS                                                                                                                                                                                                                                                                                                                                                                     

0

## Step 5 — Search the document with SQL (ranked by relevance)

The payoff. Two Db2 Text Search primitives do the work, both translated to OpenSearch under the hood:

- **`CONTAINS(chunk_text, '…') = 1`** — the predicate that filters to lexical matches.
- **`SCORE(chunk_text, '…')`** — the relevance score for each match (higher = stronger).

We order by `SCORE` and show the **top 3 full chunks**, so you see *what* matched and *how strongly*. To the application this is just SQL — no separate search API, no second datastore to keep in sync.

In [ ]:
import textwrap

def search(term, top=3):
    """Top-N lexical matches, ranked by relevance, with the FULL chunk text."""
    df = query_df(conn, f"""
        SELECT chunk_id,
               headings,
               SCORE(chunk_text, '{term}') AS RELEVANCE,
               chunk_text                  AS CHUNK_TEXT
        FROM {SCHEMA}.{TABLE}
        WHERE CONTAINS(chunk_text, '{term}') = 1
        ORDER BY RELEVANCE DESC
        FETCH FIRST {top} ROWS ONLY""")

    print(f"🔎  CONTAINS(chunk_text, {term!r})   —   top {len(df)} by relevance\n")
    for _, r in df.iterrows():
        print(f"  #{int(r.CHUNK_ID)}   relevance {r.RELEVANCE:.3f}   |  {r.HEADINGS or '(no heading)'}")
        print(textwrap.fill(str(r.CHUNK_TEXT).strip(), width=98,
                            initial_indent="     ", subsequent_indent="     "))
        print()

# A concept query — two terms, ranked by relevance, full chunk text shown.
search("vector embeddings")

In [20]:
# An exact phrase (double-quote it inside CONTAINS).
search('"external model"')

🔎  CONTAINS(chunk_text, '"external model"')   —   top 3 by relevance

  #30   relevance 0.485   |  ALTER EXTERNAL MODEL statement
     ALTER EXTERNAL MODEL statement The ALTER EXTERNAL MODEL statement updates, adds, or drops a
     metadata attribute in an existing external model.

  #52   relevance 0.473   |  EXTERNAL MODEL model-name
     EXTERNAL MODEL model-name Identifies the external model that is to have its ownership
     transferred. The model-name must identify an external model that is described in the catalog
     (SQLSTATE 42704). When ownership of the external model is transferred, the value in the OWNER
     column for the external model in the SYSCAT.EXTERNALMODELS catalog view is replaced with the
     authorization ID of the new owner.

  #56   relevance 0.472   |  ALTER
     ALTER Grants the privilege to alter existing external model by using the ALTER EXTERNAL MODEL
     statement.



In [21]:
# Boolean keyword logic.
search("model AND watsonx")

🔎  CONTAINS(chunk_text, 'model AND watsonx')   —   top 3 by relevance

  #18   relevance 0.694   |  watsonx-text-generation-options
     watsonx-text-generation-options These options can only be specified for provider WATSONX with
     model type TEXT _ GENERATION (SQLSTATE 42601).

  #43   relevance 0.691   |  watsonx-text-generation-options
     watsonx-text-generation-options These options can only be specified for provider WATSONX with
     model type TEXT _ GENERATION (SQLSTATE 42601). See CREATE EXTERNAL MODEL for more
     information.

  #8   relevance 0.625   |  URL string-constant
     URL string-constant Specifies the HTTPS endpoint of the watsonx embedding API. This endpoint
     must be a valid and accessible HTTPS URL that points to the embedding service interface of
     the model. Db2 uses this URL to issue API requests for model invocation. If watsonx supplies
     only a base URL and expects the client to construct the full endpoint dynamically, Db2
     assembles the

## Recap — and the road to hybrid search

**What we built — the lexical leg, end to end:**

```
PDF → Docling → Markdown → HybridChunker → Db2 table → Db2 Text Search (OpenSearch) → CONTAINS
```

Lexical search nails exact terms: procedure names, identifiers, error codes. But it's blind to *meaning* — a query like *"how do I turn text into vectors"* won't match a chunk that says *"generate embeddings,"* because they share no keywords.

**That gap is what the semantic leg fills. To complete hybrid search:**

1. **Embed** each chunk (`all-MiniLM-L6-v2` — the model our chunk budget already targets) and store the vectors.
2. **Vector search** for semantic / paraphrase matches.
3. **Fuse** the lexical and semantic rankings (e.g. *Reciprocal Rank Fusion*) into one ordered result list.

> Today's lexical foundation **+** the semantic leg **+** fusion **=** hybrid search.

---
## Appendix — reset for rehearsal *(optional)*

Run this to tear everything down (text index + table) for a clean re-run.

In [22]:
# Drops the text index (as db2inst1) and the table, leaving nothing behind, so
# the next full run starts from a clean slate. Optional — Step 3 already cleans
# up at the start of every run; this is just for tidy rehearsals.
run_as_db2inst1("drop_pdf_chunks_index.sh")
try:
    ibm_db.exec_immediate(conn, f"DROP TABLE {SCHEMA}.{TABLE}")
    print(f"dropped table {SCHEMA}.{TABLE}")
except Exception:
    print("table already gone")

==> Connecting to 'sample'

   Database Connection Information

 Database server        = DB2/LINUXX8664 12.1.5.0
 SQL authorization ID   = DB2INST1
 Local database alias   = SAMPLE


==> Dropping text index myschema.pdf_chunks_idx (if it exists)
CIE00001 Operation completed successfully. 
    dropped.

Done. You can now re-run chunk_and_ingest.ipynb, then ./index_pdf_chunks.sh.
dropped table myschema.pdf_chunks
